In [ ]:
# @title Setup smoke test (run this once at home before the tutorial!)
try:
    !pip install -q -r https://raw.githubusercontent.com/NeoNeuron/ccnss2026-neural-data-analysis/main/requirements.txt
    from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache  # noqa
    import ssm  # noqa
    print("✅ Setup complete.")
except Exception as e:
    print("❌ Setup FAILED:", e)
    raise

# CCNSS 2026 — Session 1: Coding and Networks

**Theme:** From single-neuron coding to network interactions.

We will analyze one Allen Visual Coding Neuropixels session in three modules:
- **1A** — Tuning curves and PSTHs
- **1B** — Signal and noise correlations
- **1C** — Functional networks and graph theory

**Time budget:** 45 minutes. Each module: ~5 min intro → 5–8 min exercise → 2 min reveal.

In [ ]:
# @title Setup (run once) { display-mode: "form" }
!pip install -q -r https://raw.githubusercontent.com/NeoNeuron/ccnss2026-neural-data-analysis/main/requirements.txt 2>&1 | tail -5
!git clone -q https://github.com/NeoNeuron/ccnss2026-neural-data-analysis.git || (cd ccnss2026-neural-data-analysis && git pull -q)
import sys; sys.path.insert(0, "/content/ccnss2026-neural-data-analysis")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ccnss_helpers import data, plotting, save_checkpoint, load_or_compute

print("✅ Setup complete.")

In [ ]:
# Loads ~2 GB Allen NWB on first run (~3-5 min). Subsequent cells re-use the cache.
session = data.load_allen_session()
print(f"Session {session['session_id']}: {len(session['units'])} good units across "
      f"{session['units']['ecephys_structure_acronym'].nunique()} areas")
session["units"].groupby("ecephys_structure_acronym").size()

In [ ]:
# Visualize ~100 simultaneous units for 5 seconds during drifting gratings.
sample_uids = session["units"].index[:100]
sample_spikes = {uid: session["spike_times"][uid] for uid in sample_uids}
t0 = session["stim_table"]["start_time"].iloc[0]
plotting.plot_raster(sample_spikes, t_start=t0, t_end=t0 + 5)
plt.title("100 simultaneously-recorded units, 5 s during drifting gratings");

## Module 1A — Tuning curves & PSTHs

**Goal:** for each unit, compute how its firing rate depends on grating orientation.
Then find the most orientation-selective neuron in the population.

**Steps:**
1. Bin spikes into a (n_units × n_trials × n_bins) array aligned to stimulus onset.
2. For each unit, compute mean rate per orientation → tuning curve.
3. Compute the orientation selectivity index (OSI):

$$\mathrm{OSI} = \frac{R_{\text{pref}} - R_{\text{orth}}}{R_{\text{pref}} + R_{\text{orth}}}$$

In [ ]:
def bin_spikes_per_trial(spike_times_dict, stim_table, bin_size_s=0.025, window_s=(0.0, 2.0)):
    """Return (n_units, n_trials, n_bins) spike counts aligned to trial start_time."""
    uids = list(spike_times_dict.keys())
    bin_edges = np.arange(window_s[0], window_s[1] + bin_size_s, bin_size_s)
    n_bins = len(bin_edges) - 1
    n_trials = len(stim_table)
    counts = np.zeros((len(uids), n_trials, n_bins), dtype=np.int32)
    for i, uid in enumerate(uids):
        st = spike_times_dict[uid]
        for j, t0 in enumerate(stim_table["start_time"].to_numpy()):
            rel = st - t0
            mask = (rel >= window_s[0]) & (rel < window_s[1])
            counts[i, j, :] = np.histogram(rel[mask], bins=bin_edges)[0]
    return counts, bin_edges, uids

counts, bin_edges, uids = bin_spikes_per_trial(
    session["spike_times"], session["stim_table"]
)
print(counts.shape)  # expected: (n_units, n_trials, n_bins)

In [ ]:
def compute_tuning_curves(counts, stim_table, bin_size_s=0.025):
    """Mean firing rate per (unit, orientation). Returns (orientations, rates[n_units, n_oris])."""
    rates_per_trial = counts.sum(axis=2) / (counts.shape[2] * bin_size_s)  # Hz
    oris = np.sort(stim_table["orientation"].dropna().unique())
    tc = np.zeros((counts.shape[0], len(oris)))
    for k, ori in enumerate(oris):
        mask = stim_table["orientation"].to_numpy() == ori
        tc[:, k] = rates_per_trial[:, mask].mean(axis=1)
    return oris, tc

oris, tc = compute_tuning_curves(counts, session["stim_table"])
print(tc.shape)

In [ ]:
# EXERCISE: compute OSI for every unit and find the most selective one
def osi(tc_row):
    """Orientation Selectivity Index for one tuning curve row."""
    pref = tc_row.max()
    pref_idx = tc_row.argmax()
    orth_idx = (pref_idx + len(tc_row) // 2) % len(tc_row)
    orth = tc_row[orth_idx]
    return (pref - orth) / (pref + orth + 1e-9)

osis = np.array([osi(row) for row in tc])
best = osis.argmax()
print(f"Most selective unit: {uids[best]} (OSI={osis[best]:.2f}, area={session['units'].loc[uids[best], 'ecephys_structure_acronym']})")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(oris, tc[best], "o-")
ax.set_xlabel("Orientation (deg)"); ax.set_ylabel("Firing rate (Hz)")
ax.set_title(f"Tuning curve, unit {uids[best]}, OSI={osis[best]:.2f}");

In [ ]:
# CUTTABLE: von Mises tuning fit
from scipy.optimize import curve_fit
def von_mises(x_deg, A, kappa, mu, offset):
    x = np.deg2rad(x_deg)
    return A * np.exp(kappa * np.cos(x - mu)) / np.exp(kappa) + offset

popt, _ = curve_fit(von_mises, oris, tc[best],
                    p0=[tc[best].max(), 2.0, np.deg2rad(oris[tc[best].argmax()]), tc[best].min()])
xs = np.linspace(0, 330, 200)
ax.plot(xs, von_mises(xs, *popt), "--", label="von Mises fit"); ax.legend(); fig

In [ ]:
save_checkpoint(
    "module_1A",
    counts=counts,
    bin_edges=bin_edges,
    uids=np.asarray(uids),
    oris=oris,
    tc=tc,
    osis=osis,
)
print("✅ 1A checkpoint saved.")

## Module 1B — Signal and noise correlations

For each pair of neurons (i, j):
- **Signal correlation:** corr of their tuning curves across orientations.
- **Noise correlation:** corr of their trial-to-trial responses at a fixed orientation, averaged across orientations.

Recreate the canonical Cohen & Kohn 2011 scatter plot: r_signal on x, r_noise on y.

In [ ]:
ck = load_or_compute(
    "module_1A",
    fn=lambda: {"error": np.array([1])},  # If we get here in 1B, 1A must have run.
)
counts, oris, tc, uids = ck["counts"], ck["oris"], ck["tc"], list(ck["uids"])

In [ ]:
def signal_correlation_matrix(tc):
    """Pearson r of tuning curves between all unit pairs."""
    return np.corrcoef(tc)

r_signal = signal_correlation_matrix(tc)
print(r_signal.shape)

In [ ]:
# EXERCISE: compute the noise-correlation matrix
def noise_correlation_matrix(counts, stim_table, bin_size_s=0.025):
    """Mean-across-orientations Pearson r of trial-to-trial residual rates."""
    rates = counts.sum(axis=2) / (counts.shape[2] * bin_size_s)  # n_units x n_trials
    rs = []
    for ori in np.sort(stim_table["orientation"].dropna().unique()):
        mask = stim_table["orientation"].to_numpy() == ori
        z = (rates[:, mask] - rates[:, mask].mean(axis=1, keepdims=True))
        # Avoid divide-by-zero for silent units.
        std = rates[:, mask].std(axis=1, keepdims=True) + 1e-9
        z = z / std
        rs.append(z @ z.T / mask.sum())
    return np.mean(rs, axis=0)

r_noise = noise_correlation_matrix(counts, session["stim_table"])

In [ ]:
# EXERCISE: make the signal-vs-noise scatter
iu = np.triu_indices_from(r_signal, k=1)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(r_signal[iu], r_noise[iu], s=2, alpha=0.3)
ax.set_xlabel("Signal correlation"); ax.set_ylabel("Noise correlation")
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
ax.set_title("Cohen & Kohn (2011), recreated");

In [ ]:
# CUTTABLE: signal-vs-noise scatter, restricted to within-area pairs in V1
v1_idx = [i for i, u in enumerate(uids) if session["units"].loc[u, "ecephys_structure_acronym"] == "VISp"]
sub_iu = np.array([(i, j) for a, i in enumerate(v1_idx) for j in v1_idx[a+1:]]).T
plt.scatter(r_signal[sub_iu[0], sub_iu[1]], r_noise[sub_iu[0], sub_iu[1]], s=2, alpha=0.5)
plt.title("V1-only pairs");

In [ ]:
save_checkpoint("module_1B", r_signal=r_signal, r_noise=r_noise, uids=np.asarray(uids))
print("✅ 1B checkpoint saved.")

## Module 1C — Functional networks & graph theory

We treat each unit as a node and connect pairs whose noise correlation exceeds a threshold.
Then we compute degree, clustering coefficient, modularity, and (challenge) communities.

In [ ]:
ck = load_or_compute("module_1B", fn=lambda: {"error": np.array([1])})
r_noise = ck["r_noise"]; uids = list(ck["uids"])
units_meta = session["units"].loc[uids]

In [ ]:
# EXERCISE: build the adjacency graph from r_noise
import networkx as nx

def build_graph(r, threshold=0.1):
    A = (np.abs(r) > threshold).astype(int)
    np.fill_diagonal(A, 0)
    return nx.from_numpy_array(A)

G = build_graph(r_noise, threshold=0.1)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# EXERCISE: compute degree + clustering, plot histograms
deg = np.array([G.degree(i) for i in range(len(uids))])
clust = np.array([nx.clustering(G, i) for i in range(len(uids))])

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(deg, bins=30); axes[0].set_xlabel("Degree")
axes[1].hist(clust, bins=30); axes[1].set_xlabel("Clustering coefficient")
plt.suptitle("Functional graph statistics");

In [ ]:
hub_idx = deg.argsort()[-5:][::-1]
print("Top 5 hubs:")
for i in hub_idx:
    print(f"  unit {uids[i]} ({units_meta.iloc[i]['ecephys_structure_acronym']}): degree={deg[i]}")

# Density per area
for area, ar in units_meta.groupby("ecephys_structure_acronym"):
    idx = [uids.index(u) for u in ar.index]
    sub = G.subgraph(idx)
    print(f"{area}: density={nx.density(sub):.3f} ({sub.number_of_nodes()} units)")

In [ ]:
# CUTTABLE: Louvain community detection + overlay vs anatomical area
from networkx.algorithms.community import louvain_communities
comms = louvain_communities(G, seed=0)
labels = np.zeros(len(uids), dtype=int)
for c, members in enumerate(comms):
    for m in members:
        labels[m] = c
plotting.plot_network(G, node_color=labels, node_size=20)
plt.title(f"Louvain communities: {len(comms)}");

In [ ]:
plotting.plot_network(G)
plt.title("Functional network (noise corr > 0.1)");